IMPORTS

In [7]:
import os, time, json, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
# Slúži na potláčanie zbytočných chýb
warnings.filterwarnings("ignore")
# Test či načítalo CUDU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupná: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.6.0+cu124
CUDA dostupná: True
GPU: NVIDIA GeForce GTX 1650
VRAM: 4.3 GB


KONFIGURÁCIA

In [ ]:
# Cesty
DATA_DIR    = "./data"
RESULTS_DIR = "./results"

# Hyperparametre
INPUT_SIZE   = 224     # VGG16 vstup
BATCH_SIZE   = 128
NUM_EPOCHS   = 10
NUM_CLASSES  = 10
VAL_SPLIT    = 0.1     # 10 % z train = validácia
SEED         = 42

# Learning rates
LR_HEAD      = 1e-3    # klasifikácia
LR_BACKBONE  = 1e-4    # konvolučné vrstvy
WEIGHT_DECAY = 1e-4

# Triedy CIFAR-10
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

torch.manual_seed(SEED)
np.random.seed(SEED)


DATASET

In [9]:
# ImageNet normalizácia - RGB normalizujeme
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transformácie
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    # Augmentácia
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(INPUT_SIZE, padding=8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

no_aug_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [10]:
def get_dataloaders(batch_size=BATCH_SIZE, val_split=VAL_SPLIT,
                    use_augmentation=True):
    # Načíta CIFAR-10 a rozdelí na train / val / test
    t_transform = train_transform if use_augmentation else no_aug_transform

    full_train = datasets.CIFAR10(root=DATA_DIR, train=True,
                                  download=True, transform=t_transform)
    test_set   = datasets.CIFAR10(root=DATA_DIR, train=False,
                                  download=True, transform=val_test_transform)

    n_val   = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    train_set, val_set = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    # Validačná sada bez augmentácie
    val_ds = copy.deepcopy(full_train)
    val_ds.transform = val_test_transform
    val_set.dataset = val_ds
    # num_workers a pin_memory -----> CUDA
    loaders = {
        "train": DataLoader(train_set, batch_size=batch_size,
                            shuffle=True,  num_workers=4, pin_memory=True), 
        "val":   DataLoader(val_set,   batch_size=batch_size,
                            shuffle=False, num_workers=4, pin_memory=True),
        "test":  DataLoader(test_set,  batch_size=batch_size,
                            shuffle=False, num_workers=4, pin_memory=True),
    }
    print(f"Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_set):,}")
    return loaders

# Načítaj dáta
loaders        = get_dataloaders(use_augmentation=True)
loaders_no_aug = get_dataloaders(use_augmentation=False)


Train: 45,000 | Val: 5,000 | Test: 10,000
Train: 45,000 | Val: 5,000 | Test: 10,000


Build VGG-16 REDEP

In [ ]:
def build_vgg16(num_classes=NUM_CLASSES, dropout=0.5):
    #Predtrénovaný VGG16 s novou klasifikačnou hlavou
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    in_features = model.classifier[6].in_features  # 4096
    model.classifier[6] = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model

# Ukáž architektúru
model_preview = build_vgg16()
print(model_preview)

# Info o parametroch
total = sum(p.numel() for p in model_preview.parameters())
print(f"Celkový počet parametrov VGG16: {total:,}")
del model_preview


Funkcia na nastavenie zmrazených vrstiev

In [ ]:
#index prvej vrstvy kazdeho bloku v model.features
BLOCK_START = {1: 0, 2: 5, 3: 10, 4: 20, 5: 30}

def set_frozen_layers(model, unfreeze_from_block=None):
    """
    unfreeze_from_block ked je:
        None   → len head       (E1 – frozen feature extraction)
        5      → block5 + head  (E2)
        4      → block4-5 +head (E3)
        3      → block3-5 + head(E4)
      'all'  → cela siet       (E5 – full fine-tuning)
    """
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    if unfreeze_from_block == "all":
        for param in model.parameters():
            param.requires_grad = True
    elif unfreeze_from_block is not None:        #ak je zadane cislo bloku
        start = BLOCK_START[unfreeze_from_block]
        for i, layer in enumerate(model.features):
            if i >= start:
                for param in layer.parameters():
                    param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trénovateľné: {trainable:,} / {total:,}  "
          f"({100*trainable/total:.1f} %)")
    return model


def get_optimizer(model):
    #AdamW s oddelenými LR pre hlavu a backbone
    head_ids        = set(id(p) for p in model.classifier.parameters())
    backbone_params = [p for p in model.parameters()
                       if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": list(model.classifier.parameters()), "lr": LR_HEAD}] #skupiny s roznymi learning rates
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


